⬅ [Examen blanc A — Fondamentaux mélangés](examen_blanc_A.ipynb) · [Examen blanc C — Problèmes moins prévisibles](examen_blanc_C.ipynb) ➡

# Examen blanc B — Tableaux et logique

**5 exercices · 50 minutes · dans l'ordre que tu veux**

C'est une simulation du test technique : les thèmes sont mélangés et les énoncés ne disent pas quelle technique utiliser. À toi de la reconnaître.

**Règles**
- Exécute la cellule juste en dessous : elle lance le chrono de 50 minutes. Chaque cellule de test affiche ensuite le temps restant.
- Travaille comme le jour du test : sans internet, sans IA et sans tes notes.
- Sous chaque exercice, la cellule ne vérifie que les **exemples de l'énoncé**. Comme dans un vrai test, des cas cachés (entrées vides, cas limites…) seront vérifiés à la fin : pense à les tester toi-même. Les exercices qui annoncent une **contrainte de taille** sont aussi testés sur de grandes entrées chronométrées : une solution juste mais trop lente ne rapporte pas le point.
- Gestion du temps : lis d'abord les 5 énoncés (3 minutes), commence par celui qui te paraît le plus simple, et ne reste pas bloqué plus de 15 minutes sur un exercice.

| Exercice | Durée indicative |
|---|---|
| 54 — Alternance de signes | 9 min |
| 55 — Centre de croix le plus lumineux | 10 min |
| 56 — Paires de valeurs séparées par un écart | 11 min |
| 57 — Ordre des tickets urgents | 9 min |
| 58 — Escalier avec pas de 1, 2 ou 3 | 11 min |

À la fin (ou quand le temps est écoulé), exécute la cellule **Correction** tout en bas. Le corrigé détaillé est dans [corriges/corrige_examen_blanc_B.ipynb](corriges/corrige_examen_blanc_B.ipynb).

In [ ]:
# ▶ Exécute cette cellule pour lancer le chrono de 50 minutes.
# Elle contient aussi les outils de test : tu n'as pas besoin de comprendre ce code.

import copy
import time

DUREE = 50 * 60
debut_examen = time.time()

def temps():
    ecoule = int(time.time() - debut_examen)
    if ecoule < DUREE:
        return f"⏱ {ecoule // 60} min écoulée(s), il reste {(DUREE - ecoule) // 60} min"
    return f"⏰ Les 50 minutes sont écoulées ({ecoule // 60} min) : lance la correction tout en bas."

def _court(texte, limite=100):
    return texte if len(texte) <= limite else texte[:limite - 3] + "..."

def verifier(fonction, cas):
    reussis = 0
    for arguments, attendu in cas:
        texte_args = _court(", ".join(repr(a) for a in arguments))
        try:
            obtenu = fonction(*copy.deepcopy(arguments))
        except Exception as erreur:
            print(f"💥 {fonction.__name__}({texte_args}) a planté : {type(erreur).__name__}: {erreur}")
            continue
        if obtenu == attendu:
            reussis += 1
            print(f"✅ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}")
        else:
            print(f"❌ {fonction.__name__}({texte_args}) -> {_court(repr(obtenu))}   (attendu : {_court(repr(attendu))})")
    print(f"\n{reussis}/{len(cas)} exemples réussis · {temps()}")

def _vitesse(fonction, grands_cas, limite=1.0):
    # Grandes entrées de la taille annoncée par l'énoncé : vérifie la complexité, pas seulement le résultat.
    total = 0.0
    for arguments, attendu in grands_cas:
        copie = copy.deepcopy(arguments)
        debut = time.perf_counter()
        try:
            resultat = fonction(*copie)
        except Exception as erreur:
            return f"plantage sur une grande entrée ({type(erreur).__name__}: {_court(str(erreur), 60)})"
        total += time.perf_counter() - debut
        if resultat != attendu:
            return f"résultat faux sur une grande entrée : {_court(repr(resultat))} (attendu : {_court(repr(attendu))})"
        if total > limite:
            return f"trop lent sur les grandes entrées (déjà {total:.1f} s, limite {limite:.0f} s) : la complexité attendue n'est pas atteinte"
    return None

def corriger(exercices):
    score = 0
    for numero, nom, cas, *grands_cas in exercices:
        fonction = globals().get(nom)
        if fonction is None:
            print(f"⏭ Exercice {numero} — {nom} : fonction introuvable (as-tu exécuté sa cellule ?)")
            continue
        ok, premier_echec = 0, None
        for arguments, attendu in cas:
            try:
                resultat = fonction(*copy.deepcopy(arguments))
                bon, obtenu = resultat == attendu, repr(resultat)
            except Exception as erreur:
                bon, obtenu = False, f"💥 {type(erreur).__name__}: {erreur}"
            if bon:
                ok += 1
            elif premier_echec is None:
                texte_args = _court(", ".join(repr(a) for a in arguments))
                premier_echec = f"{nom}({texte_args}) -> {_court(obtenu)}   (attendu : {_court(repr(attendu))})"
        if ok < len(cas):
            print(f"❌ Exercice {numero} — {nom} : {ok}/{len(cas)} tests")
            print(f"     premier échec : {premier_echec}")
            continue
        probleme = _vitesse(fonction, grands_cas[0]) if grands_cas else None
        if probleme is None:
            score += 1
            print(f"✅ Exercice {numero} — {nom} : {ok}/{len(cas)} tests" + (" + test de vitesse" if grands_cas else ""))
        else:
            print(f"⚠️ Exercice {numero} — {nom} : {ok}/{len(cas)} tests, mais {probleme}")
    print(f"\nScore : {score}/{len(exercices)} exercices entièrement réussis")
    print(temps())

print("C'est parti ! " + temps())

## Exercice 54 — Alternance de signes
*⏱ 9 min (indicatif)*

Renvoie la longueur de la plus longue portion **contiguë** dans laquelle les signes alternent strictement : chaque nombre a le signe **opposé** de celui qui le précède. La portion peut commencer par un positif (positif, négatif, positif…) ou par un négatif (négatif, positif, négatif…).

- Un `0` n'est ni positif ni négatif : il casse l'alternance et ne fait partie d'aucune portion.
- Un nombre non nul seul forme une portion de longueur 1.
- Si la liste ne contient aucun nombre non nul, renvoie `0`.

**Exemples** :
- `alternance_signes([-1, 2, -3, 4, 0, -1, 2])` → `4` (`[-1, 2, -3, 4]`)
- `alternance_signes([1, 1, 1])` → `1`

In [ ]:
def alternance_signes(nombres: list[int]) -> int:
    # À toi de jouer !
    pass


# Exemples de l'énoncé
verifier(alternance_signes, [
    (([-1, 2, -3, 4, 0, -1, 2],), 4),
    (([1, 1, 1],), 1),
])

## Exercice 55 — Centre de croix le plus lumineux
*⏱ 10 min (indicatif)*

La matrice a au moins 3 lignes et 3 colonnes. Une case **intérieure** est une case qui n'est pas sur le bord. Sa **luminosité** est la somme de sa valeur et des valeurs de ses quatre voisins directs (haut, bas, gauche, droite) : cela forme une croix.

Renvoie `[ligne, colonne]` de la case intérieure la plus lumineuse. En cas d'égalité, choisis la plus haute, puis la plus à gauche.

**Exemples** :
```
1 2 3
4 5 6
7 8 9
```
`croix_plus_forte([[1, 2, 3], [4, 5, 6], [7, 8, 9]])` → `[1, 1]` (seule case intérieure : 5 + 2 + 8 + 4 + 6 = 25)

```
0 0 0 0
0 1 0 0
0 0 2 9
0 0 0 0
```
`croix_plus_forte([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 2, 9], [0, 0, 0, 0]])` → `[2, 2]` (2 + 9 = 11 : le 9 est sur le bord, il ne peut pas être un centre, mais il compte comme voisin)

In [ ]:
def croix_plus_forte(grille: list[list[int]]) -> list[int]:
    # À toi de jouer !
    pass


# Exemples de l'énoncé
verifier(croix_plus_forte, [
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 1]),
    (([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 2, 9], [0, 0, 0, 0]],), [2, 2]),
])

## Exercice 56 — Paires de valeurs séparées par un écart
*⏱ 11 min (indicatif)*

Une liste est **triée** par ordre croissant et peut contenir des doublons. Compte les paires de **valeurs** distinctes `(x, y)` présentes dans la liste telles que `y - x == ecart` (avec `ecart ≥ 1`).

Une même paire de valeurs ne compte **qu'une fois**, même si `x` ou `y` apparaissent plusieurs fois dans la liste.

**Exemple** : `paires_ecart([1, 1, 2, 3, 4, 4], 2)` → `2` : (1, 3) et (2, 4)

**Contrainte** : jusqu'à 100 000 éléments ; traitement en O(n) attendu.

In [ ]:
def paires_ecart(nombres: list[int], ecart: int) -> int:
    # À toi de jouer !
    pass


# Exemples de l'énoncé
verifier(paires_ecart, [
    (([1, 1, 2, 3, 4, 4], 2), 2),
])

## Exercice 57 — Ordre des tickets urgents
*⏱ 9 min (indicatif)*

Chaque ticket est décrit par `[identifiant, urgence]`. Les tickets sont traités du plus urgent (urgence la plus grande) au moins urgent ; à urgence égale, le plus petit identifiant passe en premier.

Renvoie la liste des identifiants dans l'ordre de traitement.

**Exemple** : `tickets_urgents([[8, 2], [3, 5], [7, 2]])` → `[3, 7, 8]`

In [ ]:
def tickets_urgents(tickets: list[list[int]]) -> list[int]:
    # À toi de jouer !
    pass


# Exemples de l'énoncé
verifier(tickets_urgents, [
    (([[8, 2], [3, 5], [7, 2]],), [3, 7, 8]),
])

## Exercice 58 — Escalier avec pas de 1, 2 ou 3
*⏱ 11 min (indicatif)*

Pour monter un escalier de `n` marches, tu avances à chaque pas de 1, 2 ou 3 marches. Compte le nombre de suites de pas différentes qui arrivent **exactement** à la marche `n`.

L'ordre compte : `1 + 2` et `2 + 1` sont deux façons différentes. Pour `n = 0`, renvoie `1` (ne rien faire).

**Exemples** :
- `facon_monter(3)` → `4` (1 + 1 + 1, 1 + 2, 2 + 1, 3)
- `facon_monter(4)` → `7`

In [ ]:
def facon_monter(n: int) -> int:
    # À toi de jouer !
    pass


# Exemples de l'énoncé
verifier(facon_monter, [
    ((3,), 4),
    ((4,), 7),
])

## Correction
Exécute cette cellule à la fin de l'examen. Elle relance tes 5 fonctions sur des cas plus complets (cas limites compris), chronomètre celles dont l'énoncé annonce une contrainte de taille, et affiche ton score.

Ne la lis pas avant la fin : elle contient les cas cachés.

In [ ]:
corriger([
    (54, 'alternance_signes', [
        (([-1, 2, -3, 4, 0, -1, 2],), 4),
        (([1, 1, 1],), 1),
        (([],), 0),
        (([0, 0],), 0),   # que des zéros
        (([5, -5],), 2),
        (([1, -1, 1, 1, -1, 1, -1],), 4),
        (([-2, 0, 3],), 1),
    ]),
    (55, 'croix_plus_forte', [
        (([[1, 2, 3], [4, 5, 6], [7, 8, 9]],), [1, 1]),
        (([[0, 0, 0, 0], [0, 1, 0, 0], [0, 0, 2, 9], [0, 0, 0, 0]],), [2, 2]),
        (([[0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]],), [1, 1]),   # égalité partout
        (([[1, 1, 1, 1], [1, 9, 1, 1], [1, 1, 1, 1], [1, 1, 5, 1]],), [1, 1]),   # égalité : la plus haute
        (([[5, 5, 5], [5, 0, 5], [5, 5, 5]],), [1, 1]),
        (([[-1, -1, -1], [-1, -1, -1], [-1, -1, -1], [-1, -1, -1]],), [1, 1]),   # valeurs négatives
    ]),
    (56, 'paires_ecart', [
        (([1, 1, 2, 3, 4, 4], 2), 2),
        (([], 1), 0),
        (([1, 2, 3], 5), 0),
        (([1, 2, 3, 4], 1), 3),
        (([5, 5, 5, 6], 1), 1),   # les doublons ne comptent qu'une fois
        (([-3, -1, 1, 3], 2), 3),
        (([1, 3, 3, 5, 5, 7], 2), 3),
    ], [   # grandes entrées chronométrées
        (([i // 2 for i in range(20_000)], 3), 9997),
        ((list(range(0, 60_000, 3)), 6), 19998),
        ((list(range(20_000)), 20_000), 0),
        ((list(range(0, 40_000, 2)), 1), 0),
    ]),
    (57, 'tickets_urgents', [
        (([[8, 2], [3, 5], [7, 2]],), [3, 7, 8]),
        (([],), []),
        (([[1, 1]],), [1]),
        (([[5, 1], [4, 1], [3, 1]],), [3, 4, 5]),   # même urgence : par identifiant
        (([[1, 1], [2, 2], [3, 3]],), [3, 2, 1]),
        (([[10, 0], [2, 9], [7, 9], [1, 0]],), [2, 7, 1, 10]),
    ]),
    (58, 'facon_monter', [
        ((3,), 4),
        ((4,), 7),
        ((0,), 1),
        ((1,), 1),
        ((2,), 2),
        ((5,), 13),
        ((10,), 274),
        ((20,), 121415),
    ]),
])

---

**Suite :** [Examen blanc C — Problèmes moins prévisibles](examen_blanc_C.ipynb) ➡ · [Corrigé](corriges/corrige_examen_blanc_B.ipynb)